# PSEO
## Post-Secondary Employment Outcomes

Joe Maritn - joseph.martin@unc.edu
Last Updated: 10/5/2026

From the PSEO site: 
"Post-Secondary Employment Outcomes (PSEO) are experimental tabulations developed by researchers at the U.S. Census Bureau. PSEO data provide earnings and employment outcomes for college and university graduates by degree level, degree major, post-secondary institution, and state of institution. These statistics are generated by matching university transcript data with a national database of jobs, using state-of-the-art confidentiality protection mechanisms to protect the underlying data.

"The PSEO are made possible through data sharing partnerships between universities, university systems, State Departments of Education, State Labor Market Information offices, and the U.S. Census Bureau. PSEO data are available for post-secondary institutions whose transcript data have been made available to the Census Bureau through a data-sharing agreement."

In the past year, PSEO has expanded, with more institutions coming soon (according to AAU? AIR? presentation this year).

Perhaps the most frustrating part of working with this data is the lack of connection to IPEDS. Specifically, this uses different unique ID numbers, which requires some preprocessing/research before analysis. 

To use the data, go to `https://lehd.ces.census.gov/data/pseo_experimental.html` and download the file `pseoe.all.csv.gz`. This was last updated: 
**8/25/2026.**

In [1]:
# Set-Up
# Import librarys and set working directory

import pandas as pd
from os import listdir
from os.path import isfile, join

from pyprojroot import here

wd = str(here('data/raw-data/PSEO/')) # finds the directory with data
pub = str(here('data/processed-data/')) # location where processed data is stored



In [2]:
# I turned the for loop into a function. This should make it easier to create dataframes for each section of files

def create_df(path,files,var_list): # function takes working directory location (path), list of files, and list of variables
    data = pd.DataFrame(columns = var_list)
    for i in range(len(files)):
        try:
            temp = pd.read_csv(path+'/'+files[i], encoding='utf-8') # open each file and create a temp dataframe
        except: 
            temp = pd.read_csv(path+'/'+files[i], encoding='cp1252')
        temp = temp[var_list] # keep only the columns identified in the variable list
        
        temp['file'] = files[i].removesuffix('.csv').upper().removesuffix('_RV') # each row will be tied back to its original file with the file name
        
        data = pd.concat([data,temp]) # append data to the final dataframe
        
    data = data.reset_index().drop(columns='index') # reset the index 
       
    return data

In [4]:
path = wd + "\\Earnings" #set path to the Institutions folder

files = [f for f in listdir(path) if isfile(join(path, f))] #identify the file names that need to be read

In [5]:
grad_var = [
    'UNITID'
    ,'PSGRTYPE'
    ,'PGADJCT'
    ,'PGCMBAC'
    ,'NRADJCT'
    ,'NRCMBAC'
    ,'SSADJCT'
    ,'SSCMBAC'
]

# Columns to be renamed
g_rename = {
    'PSGRTYPE':'Cohort Type'
    ,'PGADJCT':'Pell Recipient Adjusted Cohort'
    ,'PGCMBAC':'Pell Recipient Graduation Rate (150% normal time)'
    ,'NRADJCT':'No Aid Adjusted Cohort'
    ,'NRCMBAC':'No Aid Graduation Rate (150% normal time)'
    ,'SSADJCT':'Direct Subsidized Loan Recipients Adjusted Cohort'
    ,'SSCMBAC':'Direct Subsidized Loan Recipients Graduation Rate (150% normal time)'
}